# <center>Automatic Number Plate Recognition System</center>

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import zipfile
from io import BytesIO
from PIL import Image
import os
# import tensorflow.lite as tflite  # For TensorFlow Lite models
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array

In [3]:
import os
print("Current working directory:", os.getcwd())

import sys
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

Current working directory: c:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\src


## Model Evaluation

In [4]:
from ultralytics import YOLO

In [12]:
# import sys
# from pathlib import Path

# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
# sys.path.append(str(Path(__file__).parent.parent.resolve()))
# sys.path.append(os.path.abspath("src"))

In [14]:
# model_path = r"models/license_plate_detector.pt"
# data_yaml = r"data/data.yaml"

model_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\models\license_plate_detector.pt"
data_yaml = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\data.yaml"

# Load your trained YOLO model
model = YOLO(model_path)

# Run validation on the dataset defined in your data.yaml file
results = model.val(data=data_yaml, imgsz=640)

# Print out the performance metrics (precision, recall, mAP, etc.)
print("Validation Results:")
print(results)

Ultralytics 8.3.87  Python-3.9.19 torch-2.4.1+cpu CPU (Intel Core(TM) i7-8665U 1.90GHz)
YOLOv8n summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


FileNotFoundError: 
Dataset 'C://Users/Dell/Desktop/Machine Learning/COMPUTER VISION/ANPR System/data/data.yaml' images not found ⚠️, missing path 'C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ALPR\data\valid\images'
Note dataset download directory is 'C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ALPR\datasets'. You can update this in 'C:\Users\Dell\AppData\Roaming\Ultralytics\settings.json'

## Load Data

In [4]:
# # Function to load images and labels
# def load_images_labels(images_path, labels_path):
#     # Ensure directories exist
#     if not os.path.exists(images_path):
#         raise FileNotFoundError(f"Directory not found: {images_path}")
#     if not os.path.exists(labels_path):
#         raise FileNotFoundError(f"Directory not found: {labels_path}")
    
#     image_files = sorted(os.listdir(images_path))
#     label_files = sorted(os.listdir(labels_path))
    
#     images = []
#     labels = []
    
#     for img_file, lbl_file in zip(image_files, label_files):
#         # Load image
#         img = Image.open(os.path.join(images_path, img_file)).convert("RGB")
#         images.append(np.array(img))
        
#         # Load label
#         with open(os.path.join(labels_path, lbl_file), 'r') as lbl:
#             label = lbl.read().strip()
#             labels.append(label)
    
#     return np.array(images), np.array(labels)

# # Paths relative to the current working directory
# train_images_path = "data/train/images"
# train_labels_path = "data/train/labels"
# test_images_path = "data/test/images"
# test_labels_path = "data/test/labels"
# valid_images_path = "data/valid/images"
# valid_labels_path = "data/valid/labels"

# # Load datasets
# train_images, train_labels = load_images_labels(train_images_path, train_labels_path)
# valid_images, valid_labels = load_images_labels(valid_images_path, valid_labels_path)
# test_images, test_labels = load_images_labels(test_images_path, test_labels_path)

# print(f"Train images: {train_images.shape}, Train labels: {len(train_labels)}")
# print(f"Validation images: {valid_images.shape}, Validation labels: {len(valid_labels)}")
# print(f"Test images: {test_images.shape}, Test labels: {len(test_labels)}")

In [5]:
# Function to load images and labels
def load_images_labels(images_path, labels_path):
    # Check if directories exist
    if not os.path.isdir(images_path):
        raise FileNotFoundError(f"Images directory not found: {images_path}")
    if not os.path.isdir(labels_path):
        raise FileNotFoundError(f"Labels directory not found: {labels_path}")
    
    image_files = sorted(os.listdir(images_path))
    label_files = sorted(os.listdir(labels_path))
    
    images, labels = [], []
    
    for img_file, lbl_file in zip(image_files, label_files):
        # Load image
        img_path = os.path.join(images_path, img_file)
        img = Image.open(img_path).convert("RGB")
        images.append(np.array(img))
        
        # Load label
        lbl_path = os.path.join(labels_path, lbl_file)
        with open(lbl_path, "r") as lbl:
            labels.append(lbl.read().strip())
    
    return np.array(images), np.array(labels)

# Absolute paths
train_images_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\train\images"
train_labels_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\train\labels"
test_images_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\test\images"
test_labels_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\test\labels"
valid_images_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\valid\images"
valid_labels_path = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\valid\labels"

# Load datasets
train_images, train_labels = load_images_labels(train_images_path, train_labels_path)
valid_images, valid_labels = load_images_labels(valid_images_path, valid_labels_path)
test_images, test_labels = load_images_labels(test_images_path, test_labels_path)

# Display dataset information
print(f"Train images: {train_images.shape}, Train labels: {len(train_labels)}")
print(f"Validation images: {valid_images.shape}, Validation labels: {len(valid_labels)}")
print(f"Test images: {test_images.shape}, Test labels: {len(test_labels)}")

Train images: (21173, 640, 640, 3), Train labels: 21173
Validation images: (2046, 640, 640, 3), Validation labels: 2046
Test images: (1018, 640, 640, 3), Test labels: 1018


## Image Resizing

In [ ]:
def resize_images(input_folder, output_folder, target_size=(640, 640)):
    """
    Resize all images in the input folder to the target size and save to the output folder.
    
    Args:
        input_folder (str): Path to the folder containing input images.
        output_folder (str): Path to the folder to save resized images.
        target_size (tuple): Target resolution for resizing (width, height).
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Loop through each image file in the input folder
    for file_name in os.listdir(input_folder):
        input_path = os.path.join(input_folder, file_name)
        
        # Check if it's an image file
        if os.path.isfile(input_path) and file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Open the image
            img = Image.open(input_path)
            
            # Resize the image
            resized_img = img.resize(target_size, Image.ANTIALIAS)
            
            # Save the resized image to the output folder
            output_path = os.path.join(output_folder, file_name)
            resized_img.save(output_path)
            print(f"Resized and saved: {output_path}")



train_images_input = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\train\images"
train_images_output = r"C:\Users\Dell\Desktop\Machine Learning\COMPUTER VISION\ANPR System\data\train\images_resized"

resize_images(train_images_input, train_images_output)
# resize_images(test_images_input, test_images_output)
# resize_images(valid_images_input, valid_images_output)